# Load modules

In [1]:
import os
import itk
import cv2
import sys
import json
import glob
import time
import imageio
import numpy as np
import pandas as pd
import nibabel as nib
import pydicom as dcm
from matplotlib import colors
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from lmfit import minimize, Parameters

from urllib.request import urlopen
from datetime import datetime as dtime



# Auxiliar Functions
## Utils

In [2]:
def getenv():
    """
    Requires sys and os modules:
    import sys
    import os
    """
    if sys.platform == 'win32':
        env_home = 'HOMEPATH'
    elif (sys.platform == 'darwin') | (sys.platform == 'linux'):
        env_home = 'HOME'
    HOMEPATH = os.getenv(env_home)
    
    return HOMEPATH

def check_path_exist(path, file=False):
    """
    Flag FILE indicates the path contains a file name (FLAG=TRUE) or the path only points to a folder (FLAG=FALSE (Default))
    """
    if file:
        is_path = os.path.isfile(path)
    else:
        is_path = os.path.isdir(path)

    print(f'{"OK:" if is_path else "ERROR:"} Path to {"file" if file else "folder"} {path} does{"" if is_path else " NOT"} exist')

    return is_path

def list_folder(path, skip_patterns=None, sorted=False):
    # Check the path exists:
    if not(os.path.isdir(path)):
        error_message = f'[ERROR]: Folder path {path} does not exist'
        sys.exit(error_message)
    
    raw_list = os.listdir(path)
    
    # Ensure patterns is a list, defaulting to ['.DS_Store']
    dsStorePattern = '.DS_Store'
    if skip_patterns is None:
        skip_patterns = [dsStorePattern]
    elif isinstance(skip_patterns, str):
        skip_patterns = [dsStorePattern, skip_patterns]  # Convert single string to list
    elif isinstance(skip_patterns, (list, tuple, set)):  
        skip_patterns = list(skip_patterns)  # Ensure it's a list
        if dsStorePattern not in skip_patterns:
            skip_patterns.append(dsStorePattern)  # Add default pattern if not already included
    else:
        raise TypeError("patterns must be a string, list, tuple, or set.")

    # dir_list = [item for item in raw_list if skip_pattern not in item]
    # any(p in item for p in skip_patterns) checks if any pattern in skip_patterns exists in item:
    dir_list = [item for item in raw_list if not any(p in item for p in skip_patterns)]

    if sorted:
        dir_list.sort()

    return dir_list


# Function to parse mixed datetime formats
def parse_mixed_datetime(ts, format='%Y%m%d%H%M%S.%f'):
    """
    # Ensure all values have milliseconds (force `.0000` if missing) before the conversion:
    """

    ts_f = ts.apply(lambda x: f"{x}.0000" if pd.notna(x) and "." not in str(x) else x)

    return pd.to_datetime(ts_f, format=format, errors="coerce")

def filter_list(lst, pattern):
    return [item for item in lst if pattern not in item]

def get_list_element(lst, pattern):
    return [item for item in lst if pattern in item]


## Enrst Angles and SPGR equations

In [3]:
def E1(TR, T1):
    
    return np.exp(-TR/T1)

def E2(TE, T2):
    
    return np.exp(-TE/T2)

def fzss(TR, T1, FA_rad):
    
    return (1.0 - E1(TR, T1))/(1.0 - np.cos(FA_rad)*E1(TR, T1))

def s0sinalfa(M0, FA_rad):

    return M0 * np.sin(FA_rad)

def st(M0, T1, TR, FA_rad, TE=np.nan, T2=np.nan):

    stT1 = s0sinalfa(M0, FA_rad) * fzss(TR, T1, FA_rad)
    
    if (np.isnan(T2)) | (np.isnan(TE)):
        return stT1
    else:
        #eq  with t2* decay
        return stT1 * E2(TE,T2)

def ernstAngle(TR, T1):
    # Flip Angle that maximises the signal for a given TR and T1 (REF: Handbook of MRI Pulse Sequences, page 587, Eq. 14.9)
    
    return np.rad2deg(np.arccos(E1(TR, T1)))

def anglesT1Maps(TR,T1):
    # Flip angles for optimal T1 Mapping with only 2 acq (REF: Deoni SCL 2003, Rapid Combined T1 and T2 mapping using..., MRM 49:515-526)
    
    f = 0.71 # see REF
    angle1 = (E1(TR, T1)*np.square(f) + (1-np.square(E1(TR, T1)))*np.sqrt(1-np.square(f)))/(1-(np.square(E1(TR, T1))*(1-np.square(f))))
    angle2 = (E1(TR, T1)*np.square(f) - (1-np.square(E1(TR, T1)))*np.sqrt(1-np.square(f)))/(1-(np.square(E1(TR, T1))*(1-np.square(f))))
    Angles = np.array([angle1,angle2])
    #return Angles
    return np.rad2deg(np.arccos(Angles))


## Helper functions for Linear T1 mapping

In [4]:
# Define a simple atomic function to perform linear fit 
# Add a check before fitting
def safe_polyfit(x, y):
    if np.any(np.isnan(x)) or np.any(np.isnan(y)):
        return np.nan, np.nan
    if np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan, np.nan
    if np.any(x == 0.0) or np.any(y == 0.0):
        return np.nan, np.nan    
    try:
        m, n = np.polyfit(x, y, deg=1)
        return m, n
    except Exception:
        return np.nan, np.nan
    
# Function to apply polyfit to a single pixel across the flip angles
def linfit_pixel(X, Y, i, j, k):
    x_vals = X[:, k, j, i]
    y_vals = Y[:, k, j, i]

    # try:
    mslope, nintercept = safe_polyfit(x_vals, y_vals)
    # except Exception:
    #     mslope = np.nan
    #     nintercept = np.nan

    return (i, j, k, mslope, nintercept)


## Optimisation functions for nonlinear T1 map estimation

In [5]:
# Define cost functions to estimate T1 with SPGR and IR
def fit_SPGR_t1(params, flip_angle_rad, values):
    s0 = params['s0'].value
    t1 = params['t1'].value
    tr = params['tr'].value
    te = params['te'].value
    t2star = params['t2star'].value

    s = st(s0, t1, tr, flip_angle_rad, te, t2star)
    
    return (np.fabs(s) - np.fabs(values))

# Define the optimisation method
def do_spgr_fit(ydata, fa_rad, tr, te=np.nan, t2star=np.nan, initial_s0=1.0, initial_t1=1000.0, T1ub = 10000.0):
    params = Parameters()
    params.add('s0', value=initial_s0, vary=True, min=0.0)
    params.add('t1', value=initial_t1, vary=True, min=0.0, max=T1ub)
    params.add('tr', value=tr, vary=False)
    params.add('te', value=te, vary=False)
    params.add('t2star', value=t2star, vary=False)
    
    result = minimize(fit_SPGR_t1, params, args=(fa_rad, ydata))

    return result    

def safe_nonlinfit(ydata, flipangles_rad, m0, t10, TR, TE, T2star, T1ub):

    if np.any(np.isnan(ydata)):
        return np.nan, np.nan
    
    if np.all(ydata == ydata[0]):
        return np.nan, np.nan
        
    try:
        nonlin_fit_output = do_spgr_fit(ydata, flipangles_rad, TR, TE, T2star, 
                                        T1ub=T1ub,
                                        initial_s0=m0, initial_t1=t10)
        return nonlin_fit_output.params['t1'].value, nonlin_fit_output.params['s0'].value    

    except Exception:
        return np.nan, np.nan
 
# Function to apply nonlinear fit to a single pixel across the flip angles
def nonlinfit_pixel(Y, FlipAngleRad, T10, TR, T1ub, i, j, k, TE=np.nan, T2star=np.nan):
    ydata = Y[:, k, j, i]

    t1hat, m0hat = safe_nonlinfit(ydata, FlipAngleRad, m0=np.max(ydata), t10=T10, TR=TR, TE=TE, T2star=T2star, T1ub=T1ub)

    return (i, j, k, t1hat, m0hat)


## T1 mapping algorithms

In [6]:
def t1fit_vfa(VFA_listOfImages, VFA_listOfFlipAngles, VFA_parameters, T10=1000.0, FitAlgo='rational_linear', flip_angle_units = 'deg', T1ub=5000.0, T1lb=100.0):
    # Algorithms can be:
    # rational_linear --> Rational Approximation (Helms et al., 2008)
    # std_linear    --> Linear approximation (Gutpa, 1977)
    # non_linear    --> Non-linear ("exact" fitting)

    # List of images must match the length of the flip angles:
    nImages = len(VFA_listOfImages)
    nFlipAngles = len(VFA_listOfFlipAngles)

    if nImages != nFlipAngles:
        print(f'[ERROR]: Length of images {nImages} does not match the list of flip angles {nFlipAngles}')
        sys.exit()

    if nImages < 2:
        print(f'[ERROR]: Not enough images {len(nImages)} to perform the calculations')
        sys.exit()

    # At least we need the TR for the fitting
    if 'TR' not in VFA_parameters.keys():
        print('[ERROR]: Parameter TR is not available, cannot perform the fitting')
        sys.exit()
    
    # If required, convert the flip angles from degree to radians (default behaviour)
    if flip_angle_units == 'deg':
        flip_angles_radians = [np.deg2rad(x) for x in VFA_listOfFlipAngles]
    else:
        flip_angles_radians = VFA_listOfFlipAngles.copy()

    # Call the fitting function 
    if FitAlgo == 'rational_linear':
        if nImages > 2:
            print('[WARNING]: Algorithm {FitAlgo} requires only 2 Flip angles, but {nImages} were provided')
            print(f'Will use only the first 2 flip angles ({flip_angles_radians[:2]})')
            nImages = 2
        T1, M0 = rational_linear_t1_fit(VFA_listOfImages[:nImages], flip_angles_radians[:nImages], VFA_parameters['TR'], T1ub=T1ub, T1lb=T1lb)
    elif FitAlgo == 'std_linear':
        T1 = 0
        M0 = 0
        T1, M0 = standard_linear_t1_fit(VFA_listOfImages[:nImages], flip_angles_radians[:nImages], VFA_parameters['TR'], T1ub=T1ub, T1lb=T1lb)
    elif FitAlgo == 'non_linear' :
        T1 = 0
        M0 = 0
        T1, M0 = nonlinear_t1_fit(VFA_listOfImages[:nImages], flip_angles_radians[:nImages], T10, VFA_parameters['TR'], T1ub=T1ub)
    else:
        print(f'[ERROR]: Method {FitAlgo} is not yet implemented')
        T1 = []
        M0 = []
        sys.exit()
    
    return T1, M0

def rational_linear_t1_fit(VFA_images, VFA_flipangles_rad, TR, T1ub, T1lb):
    
    if 0.0 in VFA_flipangles_rad:
        print('[ERROR]: One of the angles is 0, cannot perform the calculations')
        sys.exit()

    Tnum = (VFA_images[0] / VFA_flipangles_rad[0]) - (VFA_images[1] / VFA_flipangles_rad[1]) 
    Tden = (VFA_images[1] * VFA_flipangles_rad[1]) - (VFA_images[0] * VFA_flipangles_rad[0])
    Tden += 1e-6

    T1map = 2.0 * TR * Tnum / Tden

    # T1map[np.isnan(T1map)] = 1.0
    T1map[T1map > T1ub] = 0.0
    T1map[T1map < T1lb] = 0.0

    Anum = (VFA_flipangles_rad[1]/VFA_flipangles_rad[0] - VFA_flipangles_rad[0]/VFA_flipangles_rad[1])
    Aapp = VFA_images[0] * VFA_images[1] * Anum / Tden
    # Aapp[np.isnan(Aapp)] = 0.0
    Aapp[Aapp < 0.0] = 0.0

    return T1map, Aapp

def standard_linear_t1_fit(VFA_images, VFA_flipangles_rad, TR, T1ub, T1lb):


    # Create the xi and yi elements for the X and Y lists (length is the number of flip angles) and 
    # convert it to an np.array for easier manipulation (shape: flipAngle, slices, height, width)
    X = np.array([fa_image/np.tan(fa_value) for fa_image, fa_value in zip(VFA_images, VFA_flipangles_rad)])
    Y = np.array([fa_image/np.sin(fa_value) for fa_image, fa_value in zip(VFA_images, VFA_flipangles_rad)])
    if len(X.shape) == 1:
        X = X[:, np.newaxis, np.newaxis, np.newaxis]
        Y = Y[:, np.newaxis, np.newaxis, np.newaxis]
    elif len(X.shape) == 2:
        X = X[:, :, np.newaxis, np.newaxis]
        Y = Y[:, :, np.newaxis, np.newaxis]
    elif len(X.shape) == 3:
        X = X[:, :, :, np.newaxis]
        Y = Y[:, :, :, np.newaxis]
    
    nFA, nS, nH, nW = X.shape

    # Flatten all pixel coordinates (order seems counter-intuitive, but it is following ITK convention)
    pixel_indices = [(k, j, i) for k in range(nS) for j in range(nH) for i in range(nW)]

    # Run in parallel
    results = Parallel(n_jobs=-1)(delayed(linfit_pixel)(X, Y, i, j, k) for k, j, i in pixel_indices)

    # Reconstruct the output maps (slope and intercepts volumes)
    m = np.zeros((nS, nH, nW))
    n = np.zeros((nS, nH, nW))
    for i, j, k, mijk, nijk in results:
        m[k, j, i] = mijk
        n[k, j, i] = nijk
    
    T1hat = -TR / np.log(m)
    T1hat[np.isnan(T1hat)] = 0.0
    T1hat[T1hat > T1ub] = 0.0
    T1hat[T1hat < T1lb] = 0.0
    
    M0hat = n/(1-m)

    return T1hat, M0hat

def nonlinear_t1_fit(VFA_images, VFA_flipangles_rad, T10, TR, T1ub):
    
    # def nonlinfit_pixel(Y, FlipAngle, i, j, k, TR, TE=None, T2start=None):
    
    # Create the yi elements for the Y list (length is the number of flip angles) and 
    # convert it to an np.array for easier manipulation (shape: flipAngle, slices, height, width)
    Y = np.array(VFA_images)
    if len(Y.shape) == 1:
        Y = Y[:, np.newaxis, np.newaxis, np.newaxis]
    elif len(Y.shape) == 2:
        Y = Y[:, :, np.newaxis, np.newaxis]
    elif len(Y.shape) == 3:
        Y = Y[:, :, :, np.newaxis]

    nFA, nS, nH, nW = Y.shape

    # Flatten all pixel coordinates (order seems counter-intuitive, but it is following ITK convention)
    pixel_indices = [(k, j, i) for k in range(nS) for j in range(nH) for i in range(nW)]

    # Run in parallel
    results = Parallel(n_jobs=-1)(delayed(nonlinfit_pixel)(Y, VFA_flipangles_rad, T10, TR, T1ub, i, j, k) for k, j, i in pixel_indices)

    # Reconstruct the output maps (slope and intercepts volumes)
    T1hat = np.zeros((nS, nH, nW))
    M0hat = np.zeros((nS, nH, nW))

    for i, j, k, t1est, m0est in results:
        T1hat[k, j, i] = t1est
        M0hat[k, j, i] = m0est

    T1hat[np.isnan(T1hat)] = 0.0
    # T1hat[T1hat > T1ub] = 0.0
    # T1hat[T1hat < 0.0] = 0.0

    return T1hat, M0hat


## Pre-processing T1 map

In [7]:
def nt4k_bias_correction(input_itk_volume, simple=False):

    # Cast to float if needed
    input_volume = itk.cast_image_filter(input_itk_volume, ttype=[type(input_itk_volume), itk.Image[itk.F, 3]])

    # Create a mask (you can use the whole image or a thresholded region)
    mask_image = itk.Image[itk.UC, 3].New()
    mask_image.SetRegions(input_volume.GetLargestPossibleRegion())
    mask_image.CopyInformation(input_volume)
    mask_image.Allocate()
    mask_image.FillBuffer(1)  # Whole image as mask

    # Initialize the N4 bias field correction filter
    n4_filter = itk.N4BiasFieldCorrectionImageFilter.New(Input=input_volume, MaskImage=mask_image)

    if not simple:
        # # Set the shrink factor (4 is usually ok)
        n4_filter.SetNumberOfFittingLevels(4)

        # Optionally, set parameters (like number of iterations per level)
        n4_filter.SetMaximumNumberOfIterations([100, 70, 50, 30]) # for an input volume of size [120, 528, 528]

    # Update the filter to perform the correction
    n4_filter.Update()

    # Get the output image
    output_itk_volume = n4_filter.GetOutput()

    return output_itk_volume


In [8]:
def numpy_to_itk(numpy_array, reference_itk_volume):

    # Convert back to ITK image (note: GetArrayFromImage returns z,y,x => shape must match!)
    img_itk = itk.GetImageFromArray(numpy_array)

    # Copy spatial metadata from input_volume_1
    img_itk.SetOrigin(reference_itk_volume.GetOrigin())
    img_itk.SetSpacing(reference_itk_volume.GetSpacing())
    img_itk.SetDirection(reference_itk_volume.GetDirection())

    return img_itk


## Dicom Information

In [9]:
def flatten_tags_dictionary(tag_dictionary):
    # TAG_DICTIONARY is a 2-level dictionary
    # The output is a 1-d list containing the 2nd level tags 

    upper_level_tags = tag_dictionary.keys()
    flattened_tags = []
    for ul_tag in upper_level_tags:
        subsetDataLabels = list(tag_dictionary[ul_tag].keys())
        flattened_tags += subsetDataLabels

    # Add the absolute filepath to the dataframe:
    flattened_tags.append('AbsFilePath')

    return flattened_tags


In [10]:
def load_dicom_file(path_to_dicom_file, tags_to_read, read_pixel_tag=False):

    # Reads an individual DICOM file and returns the metadata and image (if required)

    dcm_object = dcm.dcmread(path_to_dicom_file, stop_before_pixels = not(read_pixel_tag))

    dcm_metadata_fields = flatten_tags_dictionary(tags_to_read)
    dcm_metadata = [None]*len(dcm_metadata_fields)

    for tagSubSetName, tagSubSetFields in tags_to_read.items():
        # print(f'TagSubSetName: {tagSubSetName}')
        for tagName, tagValue in tagSubSetFields.items():
            try:
                dcm_metadata[dcm_metadata_fields.index(tagName)] = dcm_object[tagValue['hex']].value
            except Exception:
                # print(f'Attribute {tagName} does not exist, skipping it...')
                pass
    
    # Add the filepath:
    dcm_metadata[dcm_metadata_fields.index('AbsFilePath')] = path_to_dicom_file
    dicom_object = {'metadata': dcm_metadata}

    if read_pixel_tag:
        dicom_object['image'] = dcm_object.pixel_array

    return dicom_object 


In [11]:
dicom_dictionary = {
'ID': { # Patient and Study ID
            'StudyID': {'hex': 0x00200010,
                         #  'type': str
                         },
            'PatientName': {'hex': 0x00100010,
                         #    'type': str
                            },
            'PatientID': {'hex': 0x00100020,
                         #  'type': str
                         }
     },
'SeriesID': { # Acquisitions and Series ID
            'SeriesDescription': {'hex': 0x0008103e,
                         #  'type': str
                         },
            'SeriesNumber': {'hex': 0x00200011,
                         #  'type': pd.Int64Dtype()
                         },
            'AcquisitionNumber': {'hex': 0x00200012,
                              #     'type': pd.Int64Dtype()
                                  },
            'InstanceNumber':{'hex': 0x00200013,
                              # 'type': pd.Int64Dtype()
                             }
            # 'ImagesInAcquisition': {'hex': 0x00201002,
            #                   #     'type': pd.Int64Dtype()
            #                       }
          },
'DatesTimes': { #  Dates & Times
            'StudyDate': {'hex': 0x00080020,
                         #  'type': str
                         },
            'StudyTime': {'hex': 0x00080030,
                         #  'type': str
                         },
            'SeriesDate': {'hex': 0x00080021,
                         #  'type': str
                         },
            'SeriesTime': {'hex': 0x00080031,
                         #  'type': str
                         },
            'AcquisitionDate': {'hex': 0x00080022,
                         #  'type': str
                         },
            'AcquisitionTime': {'hex': 0x00080032,
                         #  'type': str
                         },
          #   'AcquisitionDateTime': {'hex': 0x0008002A,
          #                          'type': str
          #                },
            'AcquisitionDuration': {'hex': 0x00189073, # Duration of the single continuous gathering of data over a period of time that resulted in this instance, in seconds.
                                   #  'type': pd.Float64Dtype()
                                    },
            'ContentDate': {'hex': 0x00080023,
                         #  'type': str
                         },
            'ContentTime': {'hex': 0x00080033,
                         #    'type': str
                            },
            'InstanceCreationDate': {'hex': 0x00080012,
                         #  'type': str
                         },
            'InstanceCreationTime': {'hex': 0x00080013,
                         #  'type': str
                         },
            'PerformedProcedureStepEndDate': {'hex': 0x00400250,
                         #  'type': str
                         },
            'PerformedProcedureStepEndTime': {'hex': 0x00400251,
                         #  'type': str
                         }
          },
'AcqPars': { # Acquisition Parameters
            'ImageType': {'hex': 0x00080008,
                        #   'type': 
                        },
            'ScanningSequence': {'hex': 0x00180020,
                                 },
            'SequenceVariant': {'hex': 0x00180021,
                                 },
            'ScanOptions': {'hex': 0x00180022, # Parameters of scanning sequence.
                         #    'type': str
                            },
            'MRAcquisitionType': {'hex': 0x00180023,
                              #     'type': str
                                  },
            'SequenceName': {'hex': 0x00180024, 
                         #    'type': str
                            },
            'RepetitionTime': {'hex': 0x00180080,
                              #  'type': pd.Float64Dtype()
                               },
            'EchoTime': {'hex': 0x00180081,
                         # 'type': pd.Float64Dtype()
                         },
            'FlipAngle': {'hex': 0x00181314,
                         #  'type': pd.Float64Dtype()
                          },
            'InversionTime': {'hex': 0x00180082,
                              # 'type': pd.Float64Dtype()
                         },
            'NumberOfAverages': {'hex': 0x00180083,
                              # 'type': pd.Int64Dtype()
                         },
            'NumberOfPhaseEncodingSteps': {'hex': 0x00180089,
                                        #    'type': pd.Int64Dtype()
                                           },
            'EchoTrainLength': {'hex': 0x00180091,
                              #   'type': pd.Int64Dtype()
                                },
            'PercentSampling': {'hex': 0x00180093,
                              #   'type': pd.Float64Dtype()
                                },
            'PercentPhaseFieldOfView': {'hex': 0x00180094,
                                        # 'type': pd.Float64Dtype()
                                        },
            'PixelBandwidth': {'hex': 0x00180095,
                              #  'type': pd.Float64Dtype()
                               },
            'TriggerTime': {'hex': 0x00181060, # Time, in msec, between peak of the R wave and the peak of the echo produced
                         #    'type': pd.Float64Dtype()
                            },
            'InPlanePhaseEncodingDirection': {'hex': 0x00181312,
               #   'type': str
                 },
            'TemporalPositionIdentifier': {'hex': 0x00200100,
               #   'type': str
                 },
            'NumberOfTemporalPositions': {'hex': 0x00200105,
                                        #   'type': str
                                          },
            # 'TemporalResolution': {'hex': 0x00200110,
            #                             #   'type': str
            #                               },
            'SliceLocation':{'hex': 0x00201041,
                         #     'type': pd.Float64Dtype()
                            },
            'ImagePositionPatient':{'hex': 0x00200032,
                                   #  'type': object
                                    },
            'SliceThickness': {'hex': 0x00180050,
                              #  'type': pd.Float64Dtype()
                               },
            'SpacingBetweenSlices': {'hex': 0x00180088,
                                   #   'type': pd.Float64Dtype()
                              },
            'ImageOrientationPatient': {'hex': 0x00200037,
                                        # 'type': object
                              },
          },
'ImagePars': { # Reconstruction/Image Parameters
            'AcquisitionMatrix': {'hex': 0x00181310,
                              #     'type': object
                                  },
            'PixelSpacing': {'hex': 0x00280030,
                         #     'type': object
                             },
            'Rows': {'hex': 0x00280010,
                    #  'type': pd.Int64Dtype()
                     },
            'Columns': {'hex': 0x00280011,
                    #     'type': pd.Int64Dtype()
                        },
            # 'PixelAspectRatio': {'hex': 0x00280034,
            #                   #    'type': object
            #                      },
            'ReconstructionDiameter': {'hex': 0x00181100,
                                   #     'type': pd.Float64Dtype()
                                       },
            'RescaleType': {'hex': 0x00281054,
                         #    'type': str
                            },
            'RescaleIntercept': {'hex': 0x00281052, # RI
                         #    'type': pd.Float64Dtype()
                            },
            'RescaleSlope': {'hex': 0x00281053, # RS
                         #    'type': pd.Float64Dtype()
                            },
            # 'PhilipsRWVSlope': {'hex': 0x00409225,  # Philips Private Attribute
            #                   #   'type': pd.Float64Dtype()
            #                    },
            # 'PhilipsRWVIntercept': {'hex': 0x00409224,  # WI, Philips Private Attribute
            #                        #  'type': pd.Float64Dtype()
            #                         },
            'PhilipsScaleSlope': {'hex': 0x2005100E, # SS, Philips Private Attribute
                              #     'type': pd.Float64Dtype()
                                  }
            # 'PhaseNumber': {'hex': 0x20011008, # Philips Private Attribute
            #              #    'type': pd.Int64Dtype()
            #                 }
          },
'ContrastAgent': { # Contrast Agent
            'ContrastBolusAgent': {'hex': 0x00180010,
                         #  'type': str
                         },
            'ContrastBolusRoute': {'hex': 0x00181040,
                         #  'type': str
                         },
            'ContrastBolusVolume': {'hex': 0x00181041,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastBolusStartTime': {'hex': 0x00181042,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastBolusStopTime': {'hex': 0x00181043,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastBolusTotalDose': {'hex': 0x00181044,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastFlowRate': {'hex': 0x00181046,
                         #  'type': pd.Float64Dtype()
                         },
          }
   }


# Paths and environment variables

In [12]:
HOMEPATH = getenv()
DATAPATH = os.path.join(HOMEPATH, 'Data', 'FTV_DCEMRI_Phase02')

DCMPATH = os.path.join(DATAPATH, 'DICOM')
REGPATH = os.path.join(DATAPATH, 'registration')
T1PATH = os.path.join(DATAPATH, 't1vfa')

sub_folder_list = [DCMPATH, REGPATH, T1PATH]
# Check the sub-folder exist, if not creates it:
for sub_folder in sub_folder_list:
    os.makedirs(sub_folder, exist_ok=True)
    
patient_list = list_folder(DCMPATH, sorted=True)

print(patient_list)


['001', '002', '004', '005', '006', '008', '009']


## Figures and fitting parameters

In [13]:
[fig_width, fig_height] = [8.0, 6.0]
# T1 mapping parameters
alfa_i = [5, 10, 15]
T1lb, T1ub = [200.0, 5000.0]
bin_hist = np.linspace(0.8 * T1lb, 0.8 * T1ub, num=256)
histtype = 'step' # {'bar', 'barstacked', 'step', 'stepfilled'}, default: 'bar'


# Registration Parameters
Define a parameters object using the parameters files available in the [Model Zoo](https://lkeb.ml/modelzoo/). For breast MRI, can use some of the cited in the literature:
* Par0032: [Description](https://lkeb.ml/modelzoo/par0032/) - [Parameter files](https://github.com/SuperElastix/ElastixModelZoo/tree/master/models/Par0032) 
* Par0052: [Description](https://lkeb.ml/modelzoo/par0052/) - [Parameter files](https://github.com/SuperElastix/ElastixModelZoo/tree/master/models/Par0052) 
* Par0058: [Description](https://lkeb.ml/modelzoo/par0058/) **(not yey used)** - [Parameter files](https://github.com/SuperElastix/ElastixModelZoo/tree/master/models/Par0058) 

In [14]:
# Create parameters object for a given set of parameters:
parameter_object = itk.ParameterObject.New()

# Where to save the parameters files
reg_pars_save_path = os.path.join(REGPATH, 'pars')
os.makedirs(reg_pars_save_path, exist_ok=True)

url_trunk = 'https://raw.githubusercontent.com/SuperElastix/ElastixModelZoo/refs/heads/master/models/'
parametersID = 'Par0032'
parametersFile = {'Par0032': ['Par0032_bsplines.txt', 'Par0032_rigid.txt'],
                  'Par0052': ['Elastix_Params_Affine.txt', 'Elastix_Params_BSpline.txt'],
                  'Par0058': ['Par0058trans.txt']
}

for parameters_filename in parametersFile[parametersID]:
    url_to_parameter_file = os.path.join(url_trunk, parametersID, parameters_filename)
    local_path_file = os.path.join(reg_pars_save_path, parameters_filename)

    registration_parameter_file_obj = urlopen(url_to_parameter_file)
    with open(local_path_file, "wb") as f:
        f.write(registration_parameter_file_obj.read())

    if os.path.isfile(local_path_file):
        parameter_object.AddParameterFile(local_path_file)

print(parameter_object)


ParameterObject (0x324979750)
  RTTI typeinfo:   elastix::ParameterObject
  Reference Count: 1
  Modified Time: 64
  Debug: Off
  Object Name: 
  Observers: 
    none
ParameterMap 0: 
  (BSplineInterpolationOrder 1)
  (CompressResultImage "true")
  (DefaultPixelValue 0)
  (ErodeMask "false")
  (FinalBSplineInterpolationOrder 1)
  (FinalGridSpacingInPhysicalUnits 40)
  (FixedImageDimension 3)
  (FixedImagePyramid "FixedRecursiveImagePyramid")
  (FixedInternalImagePixelType "short")
  (HowToCombineTransforms "Compose")
  (ImagePyramidSchedule 4 4 4 2 2 2 1 1 1)
  (ImageSampler "Random")
  (Interpolator "BSplineInterpolator")
  (MaximumNumberOfIterations 500)
  (Metric "AdvancedMattesMutualInformation")
  (MovingImageDimension 3)
  (MovingImagePyramid "MovingRecursiveImagePyramid")
  (MovingInternalImagePixelType "short")
  (NewSamplesEveryIteration "true")
  (NumberOfHistogramBins 32)
  (NumberOfResolutions 3)
  (NumberOfSpatialSamples 5000)
  (Optimizer "AdaptiveStochasticGradientDescen

## Pre-Processing steps

In [19]:
# Pre-Processing steps

# Noise Reduction parameters:
noiseRedMethod = 'gaussian' # None, 'gaussian' or 'median'
# Median Filter: Define the radius of the neighborhood
radius = 1
# Gaussian Filter
gvar = 2.0

noiseRedFiltFlag = False
biasFlag = True
regFlag  = True
thirdFAflag = True
# smoothFlag = False


# Loop over all datasets

In [ ]:
start_overall_time = time.perf_counter()

for patientID in patient_list[2:]: # patients 001 and 002 have different image resolution to combine VFA with DYN
    for visitNro in [1, 2]:
        study_dates = list_folder(os.path.join(DCMPATH, patientID), sorted=True)
        visits_dict = dict(zip(range(1,len(study_dates)+1), study_dates))
        visitID = visits_dict[visitNro]
        sequences_in_visit = list_folder(os.path.join(DCMPATH, patientID, visitID))

        print('From the first dicom file available, get the patient ID and study date...')
        dicom_required_fields = flatten_tags_dictionary(tag_dictionary=dicom_dictionary)
        sample_dicom_file = glob.glob(os.path.join(DCMPATH, patientID, visitID, sequences_in_visit[0], '*.dcm'))[0]
        dicom_metadata = load_dicom_file(sample_dicom_file, dicom_dictionary)

        patient_metadata = dict(zip(dicom_required_fields, dicom_metadata['metadata']))
        print('Patient Metadata:')
        print(f'\tStudy ID: {patient_metadata["StudyID"]}')
        print(f'\tPatient ID: {patient_metadata["PatientID"]}')
        print(f'\tStudyDate: {dtime.strftime(dtime.strptime(patient_metadata["StudyDate"], "%Y%m%d"), "%d/%m/%Y")}')

        print('********* Loading the FA sequences ***************')
        print('Please wait...')

        start_process_time = time.process_time()
        map_filename_pattern = f'{patient_metadata["PatientID"]}_{patientID}_{patient_metadata["StudyDate"]}'

        fixed_sequence_pattern = 'FA15'
        fixed_sequenceID = get_list_element(sequences_in_visit, fixed_sequence_pattern)[0]
        fixed_volume = itk.imread(os.path.join(DCMPATH, patientID, visitID, fixed_sequenceID), itk.F)

        moving_sequence_pattern = 'FA5'
        moving_sequenceID = get_list_element(sequences_in_visit, moving_sequence_pattern)[0]
        moving_volume = itk.imread(os.path.join(DCMPATH, patientID, visitID, moving_sequenceID), itk.F)
        print('********* Sequences loaded ***************')
        time.sleep(0.5)

        [ns, ny, nx] = fixed_volume.shape
        print(f'Fixed Volume size: {fixed_volume.shape}')
        print(f'Moving Volume size: {moving_volume.shape}')
        if not(tuple([ns, ny, nx]) == moving_volume.shape):
            print('[WARNING]: Size of the Fixed volume ({[ns, ny, nx]}) is not the same as the Moving volume ({moving_volume_raw.shape})')
        time.sleep(0.1)

        start_time = time.perf_counter()
        if noiseRedFiltFlag:
            print(f'********* Applying Noise Reduction filter ({noiseRedMethod}) *********')
            print('Please wait...')
            if noiseRedMethod is not None:
                if noiseRedMethod == 'median':
                    fixed_volume = itk.median_image_filter(fixed_volume, Radius=radius)
                    moving_volume = itk.median_image_filter(moving_volume, Radius=radius)

                elif noiseRedMethod == 'gaussian':
                    fixed_volume = itk.discrete_gaussian_image_filter(fixed_volume, variance=gvar)
                    moving_volume = itk.discrete_gaussian_image_filter(moving_volume, variance=gvar)
            else:
                print(f'No filter applied to reduce noise, NOISEREDMETHOD flag set to {noiseRedMethod}')

            end_time = time.perf_counter()
            map_filename_pattern += '_noisered'
            print(f'§§§§ Noise Reduction Filter took {(end_time - start_time):.2f}[s]')
            print('********* Noise Reduction filter applied *********')
            print(''.join(['§']*50))
            time.sleep(0.5)

        start_time = time.perf_counter()
        if biasFlag:
            print('********* Applying Bias Correction *********')
            print('Please wait...')
            fixed_volume = nt4k_bias_correction(fixed_volume)
            moving_volume = nt4k_bias_correction(moving_volume)

            end_time = time.perf_counter()
            map_filename_pattern += '_bias'
            print(f'§§§§ Bias Correction took {(end_time - start_time):.2f}[s]')
            print('********* Bias Correction applied *********')
            print(''.join(['§']*50))
            time.sleep(0.5)

        if regFlag:
            print('********* Applying Registration algorithm *********')
            print('Please wait...')
            start_time = time.perf_counter()
            moving_volume, result_transform_parameters = itk.elastix_registration_method(
                fixed_volume, moving_volume, parameter_object=parameter_object, log_to_console=False)
            end_time = time.perf_counter()
            map_filename_pattern += '_reg'
            print(f'§§§§ Registration Algorithm took {(end_time - start_time):.2f}[s]')
            print('********* Dataset registered *********')
            print(''.join(['§']*50))
            time.sleep(0.5)

        fixed_nparray = itk.GetArrayFromImage(fixed_volume)
        moving_nparray = itk.GetArrayFromImage(moving_volume)

        if thirdFAflag:
            print('********* Load the additional Flip Angle *********')
            print('Please wait...')
            start_time = time.perf_counter()
            hsrdcemri_pattern = 'Dyn'
            nt = 6
            ns = 120
            hsrdcemri_pattern_sequenceID = get_list_element(sequences_in_visit, hsrdcemri_pattern)[0]
            hsrdcemri_volume = itk.imread(os.path.join(DCMPATH, patientID, visitID, hsrdcemri_pattern_sequenceID), itk.F)
            hsrdcemri_nparray = itk.GetArrayFromImage(hsrdcemri_volume)

            # Retain only de first volume:
            moving_10deg_raw_nparray = hsrdcemri_nparray[::nt, :, :]

            # The metadata must be updated accordingly to create a valid ITK object:
            moving_volume_10deg_raw = itk.GetImageFromArray(moving_10deg_raw_nparray)

            # Get correct origin, spacing, direction
            hsrdce_spacing = list(fixed_volume.GetSpacing())  # [z, y, x]
            hsrdce_origin = list(hsrdcemri_volume.GetOrigin())    # [z, y, x]
            hsrdce_direction = hsrdcemri_volume.GetDirection()
            # Update spacing for 3D volume (slice spacing = spacing[0] * n_timepoints)
            moving_volume_10deg_raw.SetSpacing(hsrdce_spacing)
            moving_volume_10deg_raw.SetOrigin(hsrdce_origin)
            moving_volume_10deg_raw.SetDirection(hsrdce_direction)
            moving_volume_10deg_metadata = dict(moving_volume_10deg_raw)

            # Is the new dataset consistent with the Fixed Volume?
            print(f'Fixed Volume metadata: \n\t{dict(fixed_volume)}')
            print(f'Moving (10deg) Volume metadata: \n\t{dict(moving_volume_10deg_raw)}')
            print('********* Added additional flip angle *********')
            print(''.join(['§']*50))
            time.sleep(0.5)

            print('********* Apply pre-processing steps to the new flip angle *********')
            print('Please wait...')
            if biasFlag:
                moving_volume_10deg = nt4k_bias_correction(moving_volume_10deg_raw)
            else:
                moving_volume_10deg = itk.image_duplicator(moving_volume_10deg_raw)
            moving_10deg_nparray = itk.GetArrayFromImage(moving_volume_10deg)

            if regFlag:
                # Register the new dataset
                moving_volume_10deg, result_transform_10deg_parameters = itk.elastix_registration_method(
                    fixed_volume, moving_volume_10deg,
                    parameter_object=parameter_object,
                    log_to_console=False)
                moving_10deg_nparray = itk.GetArrayFromImage(moving_volume_10deg)

            end_time = time.perf_counter()
            map_filename_pattern += '_3fa'
            print(f'§§§§ Adding the additional Flip Angle Volume took {(end_time - start_time):.2f}[s]')
            print('********* Additional flip angle dataset pre-processed *********')
            print(''.join(['§']*50))
            time.sleep(0.5)

        print('********* Running the T1 fit algorith *********')
        print('It may take some time, please wait...')
        start_time = time.perf_counter()
        if thirdFAflag:
            flip_angles = {
                'images': [moving_nparray, moving_10deg_nparray, fixed_nparray],
                'deg': alfa_i
                }
            t1algorithm = 'std_linear'
        else:
            flip_angles = {
                'images': [moving_nparray, fixed_nparray],
                'deg': alfa_i[::2]
                }
            t1algorithm = 'rational_linear'
        T1map, M0map = t1fit_vfa(flip_angles['images'], flip_angles['deg'],
                                {'TR': float(patient_metadata['RepetitionTime'])},
                                FitAlgo = t1algorithm, T1ub=T1ub, T1lb=T1lb)
        end_time = time.perf_counter()
        print(f'§§§§ T1 map calculation took {(end_time - start_time):.2f}[s]')
        print('********* T1 map ready *********')
        print(''.join(['§']*50))
        time.sleep(0.5)

        print('********* Save the maps as Nifti files *********')
        print('Please wait...')
        start_time = time.perf_counter()

        itk.imwrite(numpy_to_itk(T1map, fixed_volume), os.path.join(T1PATH, f'T1map_{map_filename_pattern}.nii.gz'))
        itk.imwrite(numpy_to_itk(M0map, fixed_volume), os.path.join(T1PATH, f'M0map_{map_filename_pattern}.nii.gz'))
        end_time = time.perf_counter()
        print(f'§§§§ Saving output as NifTi file took {(end_time - start_time):.2f}[s]')
        print('********* Data saved *********')
        print(f'Finished processing data from patient {patient_metadata["PatientID"]}, visit {visitNro}({patient_metadata["StudyDate"]})')


        print('********* Smoothing Output maps and saving them *********')
        print('Please wait...')
        start_time = time.perf_counter()
        T1map = itk.median_image_filter(T1map, Radius=1)
        M0map = itk.median_image_filter(M0map, Radius=1)
        map_filename_pattern += '_smoothed'
        itk.imwrite(numpy_to_itk(T1map, fixed_volume), os.path.join(T1PATH, f'T1map_{map_filename_pattern}.nii.gz'))
        itk.imwrite(numpy_to_itk(M0map, fixed_volume), os.path.join(T1PATH, f'M0map_{map_filename_pattern}.nii.gz'))

        end_time = time.perf_counter()
        print(f'§§§§ Smoothing and savinf T1 and M0 maps took {(end_time - start_time):.2f}[s]')
        print('********* Maps ready to be saved *********')
        print(''.join(['§']*50))
        time.sleep(0.5)

        end_process_time = time.process_time()
        elp_process_time = end_process_time - start_process_time
        print(f'Finished processing data from patient {patient_metadata["PatientID"]}, visit {visitNro}({patient_metadata["StudyDate"]})')
        print(f'All done. Processing a single T1 map took {elp_process_time:.2f}[s] ({elp_process_time/60:.1f}[min])')
        print(''.join(['*']*50))
        time.sleep(0.5)

        print('Moving to the next dataset...')

end_overall_time = time.perf_counter()
elapsed_time = end_overall_time - start_overall_time
print(f'Total time to process was {elapsed_time:.2f}[s] ({(elapsed_time/60):.1f}[min])')  
print('All done. Bye!')


From the first dicom file available, get the patient ID and study date...
Patient Metadata:
	Study ID: ANON44148
	Patient ID: ANON44148
	StudyDate: 05/04/2023
********* Loading the FA sequences ***************
Please wait...
********* Sequences loaded ***************
Fixed Volume size: (120, 528, 528)
Moving Volume size: (120, 528, 528)
********* Applying Bias Correction *********
Please wait...
§§§§ Bias Correction took 276.70[s]
********* Bias Correction applied *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Applying Registration algorithm *********
Please wait...
§§§§ Registration Algorithm took 60.31[s]
********* Dataset registered *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Load the additional Flip Angle *********
Please wait...


ImageSeriesReader (0x326ea93c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:1.25174



Fixed Volume metadata: 
	{'origin': array([ -72.02628326, -149.2262471 , -171.38574484]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
Moving (10deg) Volume metadata: 
	{'origin': array([ -72.02628326, -149.2262471 , -171.38574484]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
********* Added additional flip angle *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Apply pre-processing steps to the new flip angle *********
Please wait...
§§§§ Adding the additional Flip Angle Volume took 136.78[s]
********* Additional flip angle dataset pre-processed *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Running the T1 fit algorith *********
It may take some time, please wait...


invalid value encountered in log


§§§§ T1 map calculation took 159.60[s]
********* T1 map ready *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Save the maps as Nifti files *********
Please wait...
§§§§ Saving output as NifTi file took 3.29[s]
********* Data saved *********
Finished processing data from patient ANON44148, visit 1(20230405)
********* Smoothing Output maps and saving them *********
Please wait...
§§§§ Smoothing and savinf T1 and M0 maps took 3.69[s]
********* Maps ready to be saved *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
Finished processing data from patient ANON44148, visit 1(20230405)
All done. Processing a single T1 map took 4613.54[s] (76.9[min])
**************************************************
Moving to the next dataset...
From the first dicom file available, get the patient ID and study date...
Patient Metadata:
	Study ID: ANON44148
	Patient ID: ANON44148
	StudyDate: 11/05/2023
********* Loading the FA sequences ***************
Please wait...
********* 

...

Fixed Volume metadata: 
	{'origin': array([ -80.86397552,  -92.27231911, -179.24147975]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
Moving (10deg) Volume metadata: 
	{'origin': array([ -80.86397552,  -92.27231911, -179.24147975]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
********* Added additional flip angle *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Apply pre-processing steps to the new flip angle *********
Please wait...
§§§§ Adding the additional Flip Angle Volume took 155.18[s]
********* Additional flip angle dataset pre-processed *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Running the T1 fit algorith *********
It may take some time, please wait...
§§§§ T1 map calculation took 160.32[s]
********* T1 map ready *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§

ImageSeriesReader (0x31465eb60): Non uniform sampling or missing slices detected,  maximum nonuniformity:1.25174



Fixed Volume metadata: 
	{'origin': array([ -85.77377319, -147.26233795, -174.33165815]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
Moving (10deg) Volume metadata: 
	{'origin': array([ -85.77377319, -147.26233795, -174.33165815]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
********* Added additional flip angle *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Apply pre-processing steps to the new flip angle *********
Please wait...
§§§§ Adding the additional Flip Angle Volume took 94.59[s]
********* Additional flip angle dataset pre-processed *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Running the T1 fit algorith *********
It may take some time, please wait...
§§§§ T1 map calculation took 156.08[s]
********* T1 map ready *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§

ImageSeriesReader (0x32a5dac00): Non uniform sampling or missing slices detected,  maximum nonuniformity:1.25174



Fixed Volume metadata: 
	{'origin': array([ -78.90001678, -140.38858345, -191.0250304 ]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
Moving (10deg) Volume metadata: 
	{'origin': array([ -78.90001678, -140.38858345, -191.0250304 ]), 'spacing': array([1.5       , 0.67961162, 0.67961162]), 'direction': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}
********* Added additional flip angle *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Apply pre-processing steps to the new flip angle *********
Please wait...
§§§§ Adding the additional Flip Angle Volume took 113.21[s]
********* Additional flip angle dataset pre-processed *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
********* Running the T1 fit algorith *********
It may take some time, please wait...
§§§§ T1 map calculation took 156.65[s]
********* T1 map ready *********
§§§§§§§§§§§§§§§§§§§§§§§§§§§